# H1: SSL Channel Returns vs GARCH Null

Do SSL channel test produce statistically significant directional returns compared to structureless GARCH synthetic data?

**Event definition:** A "channel test" occurs when price first enters the SSL channel zone from outside the bar's wick penetrates the HMA boundary while the previous bar remained outside. This captures genuine support/resistance tests rather than bars where price simply lingers within the channel.

**Method:** Compare mean post-test returns on real data against 1000 GARCH(1,1) synthetic series. P-value = proportion of synthetic means ≥ real mean.

In [9]:
import sys
sys.path.append('../src')
from tests import h1_test
from aggregation_data import load_and_resample
import time

# BTC/USD data
btc_4h = load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '4h')
timeframes = {
    '15min': load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '15min'),
    '30min': load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '30min'),
    '1h': load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '1h'),
    '2h': load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '2h'),
    '4h': btc_4h
}

# EUR/USD data
eur_timeframes = {
    '15min': load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '15min'),
    '30min': load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '30min'),
    '1h': load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '1h'),
    '2h': load_and_resample('../data/glbx-mdp3-20230101-20250131.ohlcv-1m.csv', '2h'),
}

# BTC FUTURES (CME) - 4H

In [ ]:
results = []
for length in [60, 120]:
    for fb in [5, 10, 20]:
        start = time.time()
        real_mean, sim_means, p_value = h1_test(btc_4h, length, fb, n_simulations=1000)
        t = time.time() - start
        print(f"SSL({length}) fwd={fb}: mean={real_mean:.6f}, p={p_value:.4f}, time={t:.0f}s")
        results.append({'length': length, 'forward_bars': fb, 'real_mean': real_mean, 'p_value': p_value})

## BTC Futures (CME) - 4H Results
1000 GARCH simulations | 387 Test events | Fresh Test definition

| SSL Length | Forward Bars | Mean Return | P-value |
|-----------|-------------|-------------|---------|
| 60 | 5 | 0.001183 | 0.1420 |
| 60 | 10 | 0.001789 | 0.1560 |
| 60 | 20 | -0.001119 | 0.6940 |
| 120 | 5 | -0.000277 | 0.6300 |
| 120 | 10 | -0.001818 | 0.8340 |
| 120 | 20 | -0.004790 | 0.9440 |

No p-value approaches significance (α = 0.05). SSL(60) shows weak positive returns at short horizons but far from significant. SSL(120) shows negative returns worse than random.

In [ ]:
for tf_name, df in timeframes.items():
    print(f"\n=== {tf_name} ({df.shape[0]} bars) ===")
    for length in [60, 120]:
        for fb in [5, 10, 20]:
            real_mean, sim_means, p_value = h1_test(df, length, fb, n_simulations=1000)
            print(f"SSL({length}) fwd={fb}: mean={real_mean:.6f}, p={p_value:.4f}")

## BTC Futures - All Timeframes Results
1000 GARCH simulations per test | Fresh Test definition

| Timeframe | SSL Length | Forward Bars | Mean Return | P-value |
|-----------|-----------|-------------|-------------|---------|
| 15min | 60 | 5 | 0.000100 | 0.4850 |
| 15min | 60 | 10 | 0.000205 | 0.5160 |
| 15min | 60 | 20 | 0.000018 | 0.5340 |
| 15min | 120 | 5 | -0.000042 | 0.4790 |
| 15min | 120 | 10 | -0.000038 | 0.5330 |
| 15min | 120 | 20 | -0.000053 | 0.5440 |
| 30min | 60 | 5 | -0.000162 | 0.7950 |
| 30min | 60 | 10 | 0.000036 | 0.4660 |
| 30min | 60 | 20 | -0.000165 | 0.6470 |
| 30min | 120 | 5 | -0.000356 | 0.9400 |
| 30min | 120 | 10 | -0.000133 | 0.6550 |
| 30min | 120 | 20 | 0.000479 | 0.1650 |
| 1h | 60 | 5 | -0.000080 | 0.6010 |
| 1h | 60 | 10 | 0.000381 | 0.2100 |
| 1h | 60 | 20 | -0.000309 | 0.6590 |
| 1h | 120 | 5 | -0.000110 | 0.6400 |
| 1h | 120 | 10 | 0.000098 | 0.4390 |
| 1h | 120 | 20 | -0.000118 | 0.5800 |
| 2h | 60 | 5 | -0.000395 | 0.7450 |
| 2h | 60 | 10 | -0.000793 | 0.8250 |
| 2h | 60 | 20 | -0.000297 | 0.6310 |
| 2h | 120 | 5 | 0.000086 | 0.4610 |
| 2h | 120 | 10 | 0.001573 | 0.0710 |
| 2h | 120 | 20 | 0.000517 | 0.3850 |

No test reaches significance. Lowest p-value: SSL(120) fwd=10 on 2H at 0.071.

In [ ]:
for tf_name, df in eur_timeframes.items():
    print(f"\n=== {tf_name} ({df.shape[0]} bars) ===")
    for length in [60, 120]:
        for fb in [5, 10, 20]:
            real_mean, sim_means, p_value = h1_test(df, length, fb, n_simulations=200)
            print(f"SSL({length}) fwd={fb}: mean={real_mean:.6f}, p={p_value:.4f}")

## EUR/USD Futures (CME) - All Timeframes Results
200 GARCH simulations per test | Fresh Test definition

| Timeframe | SSL Length | Forward Bars | Mean Return | P-value |
|-----------|-----------|-------------|-------------|---------|
| 15min | 60 | 5 | -0.000002 | 0.5100 |
| 15min | 60 | 10 | 0.000017 | 0.4750 |
| 15min | 60 | 20 | -0.000022 | 0.5150 |
| 15min | 120 | 5 | 0.000027 | 0.4700 |
| 15min | 120 | 10 | 0.000053 | 0.4450 |
| 15min | 120 | 20 | 0.000061 | 0.4250 |
| 30min | 60 | 5 | 0.000036 | 0.3950 |
| 30min | 60 | 10 | 0.000036 | 0.5000 |
| 30min | 60 | 20 | 0.000059 | 0.5400 |
| 30min | 120 | 5 | -0.000010 | 0.5450 |
| 30min | 120 | 10 | -0.000052 | 0.5350 |
| 30min | 120 | 20 | -0.000014 | 0.5250 |
| 1h | 60 | 5 | -0.000086 | 0.5250 |
| 1h | 60 | 10 | -0.000095 | 0.5300 |
| 1h | 60 | 20 | -0.000120 | 0.6200 |
| 1h | 120 | 5 | 0.000131 | 0.4600 |
| 1h | 120 | 10 | 0.000048 | 0.4600 |
| 1h | 120 | 20 | 0.000073 | 0.5650 |
| 2h | 60 | 5 | 0.000101 | 0.4550 |
| 2h | 60 | 10 | 0.000150 | 0.5100 |
| 2h | 60 | 20 | 0.000065 | 0.4650 |
| 2h | 120 | 5 | 0.000162 | 0.4800 |
| 2h | 120 | 10 | 0.000253 | 0.4650 |
| 2h | 120 | 20 | 0.000107 | 0.5500 |

All p-values cluster between 0.39-0.62. Even more uniformly null than BTC ,SSL channels show zero predictive power on EUR/USD futures.

## H1 Conclusion

**54 tests across 2 assets, 5 timeframes, 2 SSL lengths, 3 forward horizons: zero significant results.** SSL channel produce returns indistinguishable from GARCH synthetic data. The "support/resistance" effect is price interacting with its own smoothed average.